# Chapter 17 Companion Notebook: Autoencoders and Representation Learning in Business Analytics

**Book:** *Business Analytics and Artificial Intelligence: An Advanced Guide to Data-Driven Decision Making*  
**Book authors:** Hyunhwan "Aiden" Lee and Reo Song  
**Notebook author:** Hyunhwan Aiden Lee  
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indy16mm/business-analytics-ai/blob/main/notebooks/Ch17_Autoencoders_and_Representation_Learning.ipynb)

This notebook accompanies Chapter 17 of the book.

**License:** Use of this notebook is governed by the repository's
[Limited Companion Materials License](../LICENSE).




Classroom note: this notebook uses synthetic mixed-type customer data and small PyTorch models so students can run the workflow in Colab without a paid API or external dataset.

Copyright 2026 to present.

## How to use this notebook

Run the cells from top to bottom. Treat markdown sections as short lecture notes and code sections as live demos. If a section runs slowly, keep `FAST_MODE = True`, reduce `N_CUSTOMERS`, or skip the optional variant comparisons.

## Why this matters (business framing)

Business data usually contains more behavioral records than clean labels. A marketer may have millions of customer sessions, transactions, reviews, and support events, but far fewer reliable labels for churn, satisfaction, fraud, or lifetime value. Representation learning uses abundant raw data to create compact embeddings that can support many downstream decisions.

This notebook follows the practical logic of Chapter 17. We will build a synthetic mixed-type customer dataset, create a time-respecting representation pipeline, compare PCA with autoencoder embeddings, train a mixed-type decoder, inspect denoising and sparse variants, introduce a small variational autoencoder, use embeddings for segmentation and similarity search, use reconstruction error for anomaly scoring, and finish with evaluation and governance checks.

## Agenda

1. Setup and reproducibility
2. Synthetic mixed-type customer data
3. Time-respecting preprocessing for representation learning
4. Baselines first: engineered features and PCA embeddings
5. The autoencoder blueprint: encoder, bottleneck, decoder
6. Mixed-type reconstruction objectives
7. Embeddings for downstream prediction and segmentation
8. Practical variants: denoising and sparsity
9. Variational autoencoder (VAE) as a bridge to generative modeling
10. Embedding-space geometry and nearest-neighbor search
11. Reconstruction error as an anomaly score
12. Evaluation, debugging, governance, and saved artifacts
13. Exercises

## Learning objectives (measurable)

By the end of this notebook, you should be able to build a representation-learning dataset with a clear as-of time, explain why PCA is a necessary baseline for autoencoders, train a small autoencoder and extract embeddings, match reconstruction losses to continuous, binary, and categorical fields, compare frozen embeddings on a downstream task, inspect denoising and sparsity as robustness constraints, train a small VAE with beta annealing, compare Euclidean, cosine, and Mahalanobis neighbors, use reconstruction error for anomaly triage, and document a representation layer as a governed data product.

## Connection map

Earlier chapters covered PCA, supervised learning, model evaluation, and the basic deep learning training workflow. Chapter 16 added time order and sequence construction. Chapter 17 adds reusable representations. The workflow becomes: define the entity and as-of time, preprocess mixed inputs without leakage, learn an encoder from abundant data, evaluate the embedding against PCA and manual features, reuse it for downstream tasks, and monitor it as a durable business asset.

In [ ]:
# ============================================================
# 1. Setup and reproducibility
# - install missing packages if needed
# - import libraries
# - set seeds
# - configure output folders
# ============================================================
import os
import sys
import math
import json
import time
import random
import warnings
import importlib
import subprocess
from pathlib import Path


def ensure(pkg_import_name, pip_name=None):
    """Install a package only if it is missing."""
    try:
        importlib.import_module(pkg_import_name)
    except Exception:
        pip_target = pip_name or pkg_import_name
        print(f"Installing {pip_target} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_target])


for import_name, pip_name in [
    ("numpy", None),
    ("pandas", None),
    ("sklearn", "scikit-learn"),
    ("matplotlib", None),
    ("torch", None),
]:
    ensure(import_name, pip_name)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    silhouette_score,
)
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Small classroom models are faster and more stable on CPU when PyTorch avoids excessive threading.
torch.set_num_threads(1)
try:
    torch.set_num_interop_threads(1)
except RuntimeError:
    pass

warnings.filterwarnings("ignore")

SEED = 42
FAST_MODE = True
N_CUSTOMERS = 1_500 if FAST_MODE else 6_000
BATCH_SIZE = 256 if FAST_MODE else 512
TRAIN_EPOCHS = 6 if FAST_MODE else 20
VARIANT_EPOCHS = 4 if FAST_MODE else 12
VAE_EPOCHS = 5 if FAST_MODE else 15
LATENT_DIM = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BASE_DIR = Path("/content") if Path("/content").exists() else Path("/mnt/data")
OUT_DIR = BASE_DIR / "ch17_outputs"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(SEED)
print("Device:", DEVICE)
print("Output folder:", OUT_DIR)

## Utility functions

We will reuse these helpers throughout the notebook. They keep the main sections focused on modeling decisions rather than display details.

In [ ]:
# ============================================================
# Utility functions
# ============================================================

def print_section(title):
    print("=" * len(title))
    print(title)
    print("=" * len(title))


def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))


def safe_auc(y_true, prob):
    try:
        return roc_auc_score(y_true, prob)
    except Exception:
        return np.nan


def binary_metrics(y_true, prob, threshold=0.50):
    pred = (prob >= threshold).astype(int)
    return {
        "threshold": float(threshold),
        "accuracy": accuracy_score(y_true, pred),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": safe_auc(y_true, prob),
        "pred_positive_rate": float(pred.mean()),
    }


def find_best_threshold(y_true, prob, metric="f1"):
    thresholds = np.linspace(0.05, 0.95, 91)
    rows = []
    for th in thresholds:
        m = binary_metrics(y_true, prob, threshold=th)
        rows.append((th, m.get(metric, np.nan)))
    best = max(rows, key=lambda x: x[1])
    return float(best[0]), float(best[1])


def plot_confusion_matrix(y_true, prob, threshold=0.50, title="Confusion matrix"):
    pred = (prob >= threshold).astype(int)
    cm = confusion_matrix(y_true, pred)
    fig, ax = plt.subplots(figsize=(4, 4))
    im = ax.imshow(cm)
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
    ax.set_title(title)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    plt.show()


def plot_history(history, title):
    hist = pd.DataFrame(history)
    display(hist.tail())
    plt.figure(figsize=(7, 4))
    for col in hist.columns:
        if col != "epoch" and col.startswith("val_") is False:
            plt.plot(hist["epoch"], hist[col], label=col)
    for col in hist.columns:
        if col.startswith("val_"):
            plt.plot(hist["epoch"], hist[col], linestyle="--", label=col)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(title)
    plt.legend()
    plt.show()


def make_dense_one_hot_encoder(categories):
    """Return a dense OneHotEncoder that works across scikit-learn versions."""
    try:
        return OneHotEncoder(categories=categories, handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(categories=categories, handle_unknown="ignore", sparse=False)


def evaluate_binary_model_from_features(X_source, y, train_idx, val_idx, test_idx, name):
    model = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=2000, class_weight="balanced", random_state=SEED),
    )
    model.fit(X_source[train_idx], y[train_idx])
    val_prob = model.predict_proba(X_source[val_idx])[:, 1]
    test_prob = model.predict_proba(X_source[test_idx])[:, 1]
    threshold, val_score = find_best_threshold(y[val_idx], val_prob)
    metrics = binary_metrics(y[test_idx], test_prob, threshold=threshold)
    metrics["model"] = name
    metrics["val_selected_f1"] = val_score
    return metrics, test_prob, model


def plot_embedding_2d(Z, labels, title, max_points=1_200):
    rng = np.random.default_rng(SEED)
    idx = np.arange(Z.shape[0])
    if len(idx) > max_points:
        idx = rng.choice(idx, size=max_points, replace=False)
    Z2 = PCA(n_components=2, random_state=SEED).fit_transform(Z[idx])
    label_values = pd.Series(labels).iloc[idx].astype(str).values
    plt.figure(figsize=(7, 5))
    for lab in sorted(pd.unique(label_values)):
        mask = label_values == lab
        plt.scatter(Z2[mask, 0], Z2[mask, 1], s=18, alpha=0.70, label=lab)
    plt.xlabel("PCA axis 1 of embedding")
    plt.ylabel("PCA axis 2 of embedding")
    plt.title(title)
    plt.legend(loc="best", fontsize=8)
    plt.tight_layout()
    plt.show()


def summarize_results(rows):
    out = pd.DataFrame(rows).copy()
    order_cols = ["model", "threshold", "accuracy", "f1", "roc_auc", "pred_positive_rate", "val_selected_f1"]
    for col in order_cols:
        if col not in out.columns:
            out[col] = np.nan
    return out[order_cols].sort_values("roc_auc", ascending=False).reset_index(drop=True)

## 2. Synthetic mixed-type customer data

The chapter emphasizes that business records are heterogeneous. To keep the notebook shareable, we create a synthetic customer dataset that mixes continuous values, binary actions, categorical fields, missingness, and delayed labels. The setting is a subscription retail platform that wants reusable customer embeddings.

The synthetic data are designed to contain four teaching signals. First, activity and value differ across customer archetypes. Second, churn risk depends on nonlinear combinations of recency, service friction, and engagement. Third, some fields are missing for business reasons rather than at random. Fourth, a small number of unusual records are injected so reconstruction error can be used as an anomaly score later.

In [ ]:
# ============================================================
# 2.1 Synthetic schema
# ============================================================
PROFILE_NAMES = [
    "high_value_loyal",
    "coupon_sensitive",
    "new_mobile_first",
    "at_risk_service",
    "seasonal_browser",
]
PROFILE_PROBS = np.array([0.23, 0.24, 0.19, 0.16, 0.18])

CAT_LEVELS = {
    "acquisition_channel": ["search", "social", "email", "affiliate", "store"],
    "region": ["west", "midwest", "south", "northeast"],
    "device_preference": ["mobile", "desktop", "mixed"],
    "price_tier": ["budget", "mid", "premium"],
}

CONTINUOUS_RAW = [
    "monthly_spend",
    "purchase_count",
    "session_count",
    "recency_days",
    "tenure_months",
    "category_diversity",
    "avg_discount_rate",
    "return_rate",
    "support_minutes",
    "review_sentiment",
]

LOG_TRANSFORM = ["monthly_spend", "purchase_count", "session_count", "support_minutes"]

BINARY_FEATURES = [
    "email_subscriber",
    "mobile_app_user",
    "coupon_redeemer",
    "recent_complaint",
    "loyalty_member",
    "free_shipping_used",
    "returned_item",
]

CATEGORICAL_FEATURES = list(CAT_LEVELS.keys())


def choose(rng, levels, probs):
    return str(rng.choice(levels, p=np.array(probs) / np.sum(probs)))


def generate_synthetic_customers(n_customers=N_CUSTOMERS, seed=SEED):
    rng = np.random.default_rng(seed)
    rows = []
    profile_params = {
        "high_value_loyal": dict(value=125, sessions=18, recency=8, discount=0.08, friction=0.04, loyalty=0.85),
        "coupon_sensitive": dict(value=65, sessions=15, recency=14, discount=0.32, friction=0.07, loyalty=0.42),
        "new_mobile_first": dict(value=42, sessions=22, recency=10, discount=0.16, friction=0.06, loyalty=0.28),
        "at_risk_service": dict(value=75, sessions=10, recency=28, discount=0.12, friction=0.28, loyalty=0.36),
        "seasonal_browser": dict(value=38, sessions=8, recency=34, discount=0.20, friction=0.06, loyalty=0.18),
    }

    for cid in range(1, n_customers + 1):
        profile = str(rng.choice(PROFILE_NAMES, p=PROFILE_PROBS))
        p = profile_params[profile]
        as_of_month = int(rng.integers(8, 31))
        acquisition_month = int(rng.integers(1, as_of_month))
        tenure_months = max(1, as_of_month - acquisition_month)

        engagement = rng.normal(0, 1)
        service_shock = rng.random() < p["friction"]
        loyalty_member = rng.random() < p["loyalty"]
        mobile_app_user = rng.random() < (0.78 if profile == "new_mobile_first" else 0.45)
        coupon_redeemer = rng.random() < (0.70 if profile == "coupon_sensitive" else 0.25)
        email_subscriber = rng.random() < (0.72 if profile in ["high_value_loyal", "coupon_sensitive"] else 0.45)

        monthly_spend = rng.lognormal(mean=np.log(p["value"]), sigma=0.40) * (1 + 0.04 * engagement)
        monthly_spend = max(monthly_spend, 2.0)
        purchase_count = rng.poisson(max(0.2, monthly_spend / 35 + (0.7 if loyalty_member else 0)))
        session_count = rng.poisson(max(2, p["sessions"] + 2.5 * engagement + 4 * mobile_app_user))
        recency_days = int(np.clip(rng.normal(p["recency"], 9), 0, 90))
        category_diversity = int(np.clip(rng.poisson(2 + monthly_spend / 70 + loyalty_member), 1, 16))
        avg_discount_rate = float(np.clip(rng.normal(p["discount"], 0.08), 0, 0.75))
        recent_complaint = rng.random() < (0.18 + 0.45 * service_shock)
        returned_item = rng.random() < (0.08 + 0.20 * recent_complaint)
        return_rate = float(np.clip(rng.beta(1.2 + 3 * returned_item, 18), 0, 0.85))
        support_minutes = rng.exponential(5 + 45 * recent_complaint + 18 * service_shock)
        review_sentiment = float(np.clip(rng.normal(0.55 - 0.45 * recent_complaint + 0.12 * loyalty_member, 0.22), -1, 1))
        free_shipping_used = rng.random() < min(0.85, 0.20 + monthly_spend / 180)

        acquisition_channel = choose(
            rng,
            CAT_LEVELS["acquisition_channel"],
            [0.35, 0.18, 0.25, 0.10, 0.12] if profile != "new_mobile_first" else [0.22, 0.37, 0.15, 0.08, 0.18],
        )
        region = choose(rng, CAT_LEVELS["region"], [0.36, 0.20, 0.27, 0.17])
        device_preference = choose(
            rng,
            CAT_LEVELS["device_preference"],
            [0.72, 0.10, 0.18] if mobile_app_user else [0.18, 0.58, 0.24],
        )
        if monthly_spend > 110:
            price_tier = choose(rng, CAT_LEVELS["price_tier"], [0.10, 0.42, 0.48])
        elif coupon_redeemer:
            price_tier = choose(rng, CAT_LEVELS["price_tier"], [0.55, 0.38, 0.07])
        else:
            price_tier = choose(rng, CAT_LEVELS["price_tier"], [0.28, 0.55, 0.17])

        sentiment_for_label = review_sentiment
        churn_logit = (
            -2.65
            + 0.035 * recency_days
            + 0.95 * recent_complaint
            + 0.35 * returned_item
            + 0.013 * support_minutes
            - 0.55 * loyalty_member
            - 0.010 * monthly_spend
            - 0.60 * sentiment_for_label
            + 0.45 * (profile == "at_risk_service")
            + 0.25 * (profile == "seasonal_browser")
        )
        churn_next_60d = rng.random() < sigmoid_np(churn_logit)

        cross_sell_logit = (
            -1.35
            + 0.012 * monthly_spend
            + 0.22 * category_diversity
            + 0.45 * loyalty_member
            + 0.30 * mobile_app_user
            - 0.60 * recent_complaint
            + 0.20 * (price_tier == "premium")
        )
        cross_sell_next_60d = rng.random() < sigmoid_np(cross_sell_logit)

        rows.append({
            "customer_id": cid,
            "as_of_month": as_of_month,
            "customer_archetype": profile,
            "monthly_spend": monthly_spend,
            "purchase_count": purchase_count,
            "session_count": session_count,
            "recency_days": recency_days,
            "tenure_months": tenure_months,
            "category_diversity": category_diversity,
            "avg_discount_rate": avg_discount_rate,
            "return_rate": return_rate,
            "support_minutes": support_minutes,
            "review_sentiment": review_sentiment,
            "email_subscriber": int(email_subscriber),
            "mobile_app_user": int(mobile_app_user),
            "coupon_redeemer": int(coupon_redeemer),
            "recent_complaint": int(recent_complaint),
            "loyalty_member": int(loyalty_member),
            "free_shipping_used": int(free_shipping_used),
            "returned_item": int(returned_item),
            "acquisition_channel": acquisition_channel,
            "region": region,
            "device_preference": device_preference,
            "price_tier": price_tier,
            "churn_next_60d": int(churn_next_60d),
            "cross_sell_next_60d": int(cross_sell_next_60d),
            "anomaly_label": 0,
        })

    df = pd.DataFrame(rows)

    # Business-style missingness, not purely random missingness.
    review_missing = (rng.random(len(df)) < 0.20) | ((df["purchase_count"] <= 1) & (rng.random(len(df)) < 0.35))
    discount_missing = (rng.random(len(df)) < 0.07) | ((df["coupon_redeemer"] == 0) & (rng.random(len(df)) < 0.08))
    df.loc[review_missing, "review_sentiment"] = np.nan
    df.loc[discount_missing, "avg_discount_rate"] = np.nan

    # Inject unusual records for the anomaly detection section.
    n_anom = max(20, int(0.025 * len(df)))
    anom_idx = rng.choice(df.index.to_numpy(), size=n_anom, replace=False)
    df.loc[anom_idx, "monthly_spend"] *= rng.uniform(3.0, 7.0, size=n_anom)
    df.loc[anom_idx, "support_minutes"] += rng.uniform(80, 220, size=n_anom)
    df.loc[anom_idx, "return_rate"] = np.clip(df.loc[anom_idx, "return_rate"] + rng.uniform(0.35, 0.70, size=n_anom), 0, 0.95)
    df.loc[anom_idx, "recent_complaint"] = 1
    df.loc[anom_idx, "anomaly_label"] = 1

    return df


customers_df = generate_synthetic_customers()
print("Shape:", customers_df.shape)
display(customers_df.head())

In [ ]:
# ============================================================
# 2.2 Quick data checks
# ============================================================
print("Customer archetype summary:")
summary = (
    customers_df.groupby("customer_archetype")
    .agg(
        n=("customer_id", "size"),
        churn_rate=("churn_next_60d", "mean"),
        cross_sell_rate=("cross_sell_next_60d", "mean"),
        anomaly_rate=("anomaly_label", "mean"),
        median_monthly_spend=("monthly_spend", "median"),
        median_recency_days=("recency_days", "median"),
    )
    .reset_index()
)
display(summary)

missing_summary = customers_df[CONTINUOUS_RAW + BINARY_FEATURES + CATEGORICAL_FEATURES].isna().mean().reset_index()
missing_summary.columns = ["field", "missing_rate"]
print("Missingness summary:")
display(missing_summary.sort_values("missing_rate", ascending=False).head(10))

plt.figure(figsize=(7, 4))
plt.hist(customers_df["monthly_spend"], bins=40)
plt.title("Raw monthly spend is skewed")
plt.xlabel("monthly_spend")
plt.ylabel("Number of customers")
plt.show()

plt.figure(figsize=(7, 4))
plt.hist(customers_df["as_of_month"], bins=23)
plt.title("Distribution of as-of months")
plt.xlabel("as-of month")
plt.ylabel("Number of records")
plt.show()

## 3. Time-respecting preprocessing for representation learning

A reusable representation must have a clear decision contract. In this notebook, the entity is a customer, the as-of time is `as_of_month`, the input is the customer record available at that month, and downstream labels such as churn or cross-sell occur after the as-of time. Even when the model is unsupervised, preprocessing should be fit on the training period only and then applied forward.

The preprocessing step also reflects Chapter 17's mixed-data argument. We log-transform skewed positive variables, standardize continuous variables, carry missingness indicators, keep binary fields as binary, and one-hot encode categorical fields.

In [ ]:
# ============================================================
# 3.1 Feature engineering with explicit as-of logic
# ============================================================
model_df = customers_df.copy()
for col in LOG_TRANSFORM:
    model_df[f"log_{col}"] = np.log1p(model_df[col])

CONTINUOUS_FEATURES = []
for col in CONTINUOUS_RAW:
    CONTINUOUS_FEATURES.append(f"log_{col}" if col in LOG_TRANSFORM else col)

# Time-respecting split. The model sees earlier months first and is evaluated later.
train_idx = model_df.index[model_df["as_of_month"] <= 18].to_numpy()
val_idx = model_df.index[(model_df["as_of_month"] > 18) & (model_df["as_of_month"] <= 24)].to_numpy()
test_idx = model_df.index[model_df["as_of_month"] > 24].to_numpy()

y_churn = model_df["churn_next_60d"].to_numpy(dtype=np.int64)
y_cross_sell = model_df["cross_sell_next_60d"].to_numpy(dtype=np.int64)
y_anomaly = model_df["anomaly_label"].to_numpy(dtype=np.int64)

split_rows = []
for name, idx in [("train", train_idx), ("val", val_idx), ("test", test_idx)]:
    split_rows.append({
        "split": name,
        "n": len(idx),
        "min_as_of_month": int(model_df.iloc[idx]["as_of_month"].min()),
        "max_as_of_month": int(model_df.iloc[idx]["as_of_month"].max()),
        "churn_rate": float(y_churn[idx].mean()),
        "cross_sell_rate": float(y_cross_sell[idx].mean()),
        "anomaly_rate": float(y_anomaly[idx].mean()),
    })

split_df = pd.DataFrame(split_rows)
display(split_df)

In [ ]:
# ============================================================
# 3.2 Fit preprocessing only on the training period
# ============================================================

def fit_representation_preprocessor(df, train_idx):
    train_df = df.iloc[train_idx].copy()
    medians = train_df[CONTINUOUS_FEATURES].median()
    scaler = StandardScaler()
    scaler.fit(train_df[CONTINUOUS_FEATURES].fillna(medians))

    categories = [CAT_LEVELS[col] for col in CATEGORICAL_FEATURES]
    cat_encoder = make_dense_one_hot_encoder(categories=categories)
    cat_encoder.fit(train_df[CATEGORICAL_FEATURES])

    cat_maps = {
        col: {level: i for i, level in enumerate(CAT_LEVELS[col])}
        for col in CATEGORICAL_FEATURES
    }

    return {
        "medians": medians,
        "scaler": scaler,
        "cat_encoder": cat_encoder,
        "cat_maps": cat_maps,
    }


def transform_representation_inputs(df, prep):
    cont_raw = df[CONTINUOUS_FEATURES].copy()
    cont_mask = (~cont_raw.isna()).astype(np.float32).to_numpy()
    cont_missing = cont_raw.isna().astype(np.float32).to_numpy()
    cont_imputed = cont_raw.fillna(prep["medians"])
    cont_scaled = prep["scaler"].transform(cont_imputed).astype(np.float32)

    binary_arr = df[BINARY_FEATURES].astype(np.float32).to_numpy()
    cat_onehot = prep["cat_encoder"].transform(df[CATEGORICAL_FEATURES]).astype(np.float32)

    X = np.concatenate([cont_scaled, cont_missing, binary_arr, cat_onehot], axis=1).astype(np.float32)

    cat_targets = []
    for col in CATEGORICAL_FEATURES:
        mapped = df[col].map(prep["cat_maps"][col]).fillna(0).astype(np.int64).to_numpy()
        cat_targets.append(mapped)
    cat_targets = np.vstack(cat_targets).T.astype(np.int64)

    slices = {}
    start = 0
    slices["continuous_scaled"] = slice(start, start + len(CONTINUOUS_FEATURES))
    start += len(CONTINUOUS_FEATURES)
    slices["missing_indicators"] = slice(start, start + len(CONTINUOUS_FEATURES))
    start += len(CONTINUOUS_FEATURES)
    slices["binary"] = slice(start, start + len(BINARY_FEATURES))
    start += len(BINARY_FEATURES)
    slices["categorical_onehot"] = slice(start, start + cat_onehot.shape[1])

    return {
        "X": X,
        "cont_scaled": cont_scaled,
        "cont_mask": cont_mask,
        "cont_missing": cont_missing,
        "binary": binary_arr,
        "cat_onehot": cat_onehot,
        "cat_targets": cat_targets,
        "slices": slices,
    }


preprocessor = fit_representation_preprocessor(model_df, train_idx)
arrays = transform_representation_inputs(model_df, preprocessor)
X_rep = arrays["X"]

print("Preprocessed representation matrix shape:", X_rep.shape)
print("Continuous features:", CONTINUOUS_FEATURES)
print("Binary features:", BINARY_FEATURES)
print("Categorical features:", CATEGORICAL_FEATURES)
print("Any missing values in X_rep?", bool(np.isnan(X_rep).any()))

In [ ]:
# ============================================================
# 3.3 Why scaling and masks matter
# ============================================================
raw_scale_demo = pd.DataFrame({
    "feature": CONTINUOUS_RAW,
    "raw_std": [customers_df[col].std(skipna=True) for col in CONTINUOUS_RAW],
    "missing_rate": [customers_df[col].isna().mean() for col in CONTINUOUS_RAW],
})
raw_scale_demo["rank_by_raw_std"] = raw_scale_demo["raw_std"].rank(ascending=False)
display(raw_scale_demo.sort_values("raw_std", ascending=False))

plt.figure(figsize=(8, 4))
plt.bar(raw_scale_demo.sort_values("raw_std", ascending=False)["feature"], raw_scale_demo.sort_values("raw_std", ascending=False)["raw_std"])
plt.xticks(rotation=35, ha="right")
plt.ylabel("Raw standard deviation")
plt.title("Unscaled features can dominate reconstruction loss")
plt.tight_layout()
plt.show()

## 4. Baselines first: engineered features and PCA embeddings

Chapter 17 emphasizes that an autoencoder should not be accepted just because reconstruction loss goes down. A strong baseline is required. PCA is especially important because a linear autoencoder trained with squared error on centered data recovers the same subspace as PCA. If the nonlinear autoencoder does not improve downstream evidence relative to PCA or engineered features, the extra complexity is not justified.

In [ ]:
# ============================================================
# 4.1 Engineered feature and PCA baselines for the anchor task
# ============================================================
representation_results = []

engineered_metrics, engineered_prob, engineered_model = evaluate_binary_model_from_features(
    X_rep, y_churn, train_idx, val_idx, test_idx, name="engineered_features_logistic"
)
representation_results.append(engineered_metrics)

pca = PCA(n_components=LATENT_DIM, random_state=SEED)
Z_pca = np.zeros((X_rep.shape[0], LATENT_DIM), dtype=np.float32)
Z_pca[train_idx] = pca.fit_transform(X_rep[train_idx]).astype(np.float32)
Z_pca[val_idx] = pca.transform(X_rep[val_idx]).astype(np.float32)
Z_pca[test_idx] = pca.transform(X_rep[test_idx]).astype(np.float32)

pca_metrics, pca_prob, pca_model = evaluate_binary_model_from_features(
    Z_pca, y_churn, train_idx, val_idx, test_idx, name="PCA_embedding_logistic"
)
representation_results.append(pca_metrics)

print("PCA explained variance ratio by component:")
display(pd.DataFrame({
    "component": np.arange(1, LATENT_DIM + 1),
    "explained_variance_ratio": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_),
}))

baseline_results_df = summarize_results(representation_results)
display(baseline_results_df)
plot_confusion_matrix(y_churn[test_idx], engineered_prob, threshold=engineered_metrics["threshold"], title="Engineered feature baseline")

In [ ]:
# ============================================================
# 4.2 Visualize PCA embeddings
# ============================================================
plot_embedding_2d(
    Z_pca,
    labels=model_df["customer_archetype"],
    title="PCA embedding map by synthetic customer archetype",
)

## 5. The autoencoder blueprint

An autoencoder has an encoder, a latent code, and a decoder. The encoder maps the input record `x` to a compact representation `z`. The decoder maps `z` back to a reconstruction `x_hat`. The bottleneck and regularization prevent the model from simply copying the input.

The first model below uses a standard multilayer perceptron and a single mean-squared reconstruction objective over the preprocessed feature vector. This is a useful blueprint demo, but the next section will improve the objective for mixed data types.

In [ ]:
# ============================================================
# 5.1 Draw a simple autoencoder blueprint
# ============================================================
def draw_autoencoder_blueprint():
    fig, ax = plt.subplots(figsize=(9, 2.6))
    ax.axis("off")
    boxes = [
        (0.08, 0.38, 0.18, 0.28, "Input x"),
        (0.32, 0.34, 0.20, 0.36, "Encoder"),
        (0.58, 0.42, 0.14, 0.20, "Latent z"),
        (0.78, 0.34, 0.20, 0.36, "Decoder"),
    ]
    for x, y, w, h, label in boxes:
        ax.add_patch(plt.Rectangle((x, y), w, h, fill=False, linewidth=2))
        ax.text(x + w / 2, y + h / 2, label, ha="center", va="center", fontsize=12)
    arrow_pairs = [(0.26, 0.50, 0.32, 0.50), (0.52, 0.50, 0.58, 0.50), (0.72, 0.50, 0.78, 0.50)]
    for x1, y1, x2, y2 in arrow_pairs:
        ax.annotate("", xy=(x2, y2), xytext=(x1, y1), arrowprops=dict(arrowstyle="->", linewidth=2))
    ax.text(0.88, 0.18, "x_hat", ha="center", va="center", fontsize=12)
    ax.annotate("", xy=(0.88, 0.34), xytext=(0.88, 0.23), arrowprops=dict(arrowstyle="->", linewidth=1.5))
    ax.text(0.50, 0.12, "Training signal: reconstruction loss between x and x_hat", ha="center", fontsize=11)
    ax.set_title("Autoencoder blueprint")
    plt.show()


draw_autoencoder_blueprint()

In [ ]:
# ============================================================
# 5.2 PyTorch Dataset and DataLoaders
# ============================================================
class RepresentationDataset(Dataset):
    def __init__(self, arrays, indices):
        self.X = torch.as_tensor(arrays["X"][indices], dtype=torch.float32)
        self.cont = torch.as_tensor(arrays["cont_scaled"][indices], dtype=torch.float32)
        self.cont_mask = torch.as_tensor(arrays["cont_mask"][indices], dtype=torch.float32)
        self.binary = torch.as_tensor(arrays["binary"][indices], dtype=torch.float32)
        self.cat = torch.as_tensor(arrays["cat_targets"][indices], dtype=torch.long)
        self.indices = np.array(indices)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return {
            "X": self.X[idx],
            "cont": self.cont[idx],
            "cont_mask": self.cont_mask[idx],
            "binary": self.binary[idx],
            "cat": self.cat[idx],
            "row_index": int(self.indices[idx]),
        }


def make_loaders(batch_size=BATCH_SIZE):
    train_loader = DataLoader(RepresentationDataset(arrays, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(RepresentationDataset(arrays, val_idx), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(RepresentationDataset(arrays, test_idx), batch_size=batch_size, shuffle=False)
    all_loader = DataLoader(RepresentationDataset(arrays, np.arange(len(model_df))), batch_size=batch_size, shuffle=False)
    return train_loader, val_loader, test_loader, all_loader


def move_batch(batch, device=DEVICE):
    return {
        key: value.to(device) if torch.is_tensor(value) else value
        for key, value in batch.items()
    }


train_loader, val_loader, test_loader, all_loader = make_loaders()
print("Number of batches:", {"train": len(train_loader), "val": len(val_loader), "test": len(test_loader)})

In [ ]:
# ============================================================
# 5.3 Basic autoencoder model and training helpers
# ============================================================
class BasicAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=LATENT_DIM, hidden_dim=64, dropout=0.10):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, latent_dim),
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decoder(z)
        return x_hat, z


def train_basic_autoencoder(model, train_loader, val_loader, epochs=TRAIN_EPOCHS, lr=1e-3, name="basic_autoencoder"):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for batch in train_loader:
            batch = move_batch(batch)
            x = batch["X"]
            optimizer.zero_grad()
            x_hat, _ = model(x)
            loss = criterion(x_hat, x)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.detach().cpu()))

        model.eval()
        val_losses = []
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch)
                x = batch["X"]
                x_hat, _ = model(x)
                loss = criterion(x_hat, x)
                val_losses.append(float(loss.detach().cpu()))

        history.append({"epoch": epoch, "train_loss": np.mean(train_losses), "val_loss": np.mean(val_losses)})
        if epoch in [1, epochs] or epoch % max(1, epochs // 3) == 0:
            print(f"{name} epoch {epoch:02d} | train {np.mean(train_losses):.4f} | val {np.mean(val_losses):.4f}")

    return history


@torch.no_grad()
def encode_basic_model(model, X, batch_size=512):
    model.eval()
    zs = []
    recons = []
    for start in range(0, len(X), batch_size):
        x = torch.as_tensor(X[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        x_hat, z = model(x)
        zs.append(z.detach().cpu().numpy())
        recons.append(x_hat.detach().cpu().numpy())
    return np.vstack(zs), np.vstack(recons)

In [ ]:
# ============================================================
# 5.4 Train the basic autoencoder and extract embeddings
# ============================================================
input_dim = X_rep.shape[1]
basic_ae = BasicAutoencoder(input_dim=input_dim, latent_dim=LATENT_DIM)
basic_history = train_basic_autoencoder(basic_ae, train_loader, val_loader, epochs=TRAIN_EPOCHS, lr=1e-3, name="Basic AE")
plot_history(basic_history, "Basic autoencoder reconstruction loss")

Z_basic, X_hat_basic = encode_basic_model(basic_ae, X_rep)
basic_recon_mse = np.mean((X_hat_basic - X_rep) ** 2, axis=1)

basic_metrics, basic_prob, basic_downstream_model = evaluate_binary_model_from_features(
    Z_basic, y_churn, train_idx, val_idx, test_idx, name="basic_AE_embedding_logistic"
)
representation_results.append(basic_metrics)

print("Basic AE reconstruction error summary:")
display(pd.Series(basic_recon_mse).describe().to_frame("mse"))
print("Downstream churn metrics using frozen basic AE embeddings:")
display(pd.DataFrame([basic_metrics]))
plot_embedding_2d(Z_basic, labels=model_df["customer_archetype"], title="Basic autoencoder embeddings by archetype")

In [ ]:
# ============================================================
# 5.5 Reconstruction error by input group
# ============================================================
def group_mse_summary(X_true, X_hat, slices, indices):
    rows = []
    for group_name, sl in slices.items():
        err = np.mean((X_true[indices, sl] - X_hat[indices, sl]) ** 2)
        rows.append({"group": group_name, "mse": float(err), "n_columns": sl.stop - sl.start})
    return pd.DataFrame(rows).sort_values("mse", ascending=False)

print("Basic AE test reconstruction error by feature group:")
display(group_mse_summary(X_rep, X_hat_basic, arrays["slices"], test_idx))

## 6. Mixed-type reconstruction objectives

A single MSE loss over a preprocessed vector is convenient, but it hides the fact that business data contains different feature types. Chapter 17 recommends a multi-head decoder: one encoder produces `z`, while decoder heads reconstruct continuous, binary, and categorical fields with losses that match each type.

In this section, the continuous head uses masked MSE so missing values do not contribute to the reconstruction target, the binary head uses binary cross-entropy, and the categorical heads use cross-entropy.

In [ ]:
# ============================================================
# 6.1 Reconstruction objective map
# ============================================================
loss_map = pd.DataFrame([
    {
        "feature_type": "continuous",
        "examples": "standardized spend, recency, tenure, rates",
        "decoder_output": "linear values",
        "loss": "masked MSE",
        "why": "missing continuous values should not be treated as zero targets",
    },
    {
        "feature_type": "binary",
        "examples": "opened, complained, coupon used, loyalty member",
        "decoder_output": "logits converted to probabilities",
        "loss": "binary cross-entropy",
        "why": "binary outcomes are naturally reconstructed as probabilities",
    },
    {
        "feature_type": "categorical",
        "examples": "region, device, channel, price tier",
        "decoder_output": "one logit vector per categorical field",
        "loss": "categorical cross-entropy",
        "why": "categories are mutually exclusive choices within each field",
    },
])
display(loss_map)

In [ ]:
# ============================================================
# 6.2 Multi-head mixed-type autoencoder
# ============================================================
class MixedTypeAutoencoder(nn.Module):
    def __init__(self, input_dim, n_cont, n_binary, cat_cardinalities, latent_dim=LATENT_DIM, hidden_dim=64, dropout=0.10):
        super().__init__()
        self.encoder_net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, latent_dim),
        )
        self.shared_decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
        )
        self.cont_head = nn.Linear(hidden_dim, n_cont)
        self.bin_head = nn.Linear(hidden_dim, n_binary)
        self.cat_heads = nn.ModuleList([nn.Linear(hidden_dim, k) for k in cat_cardinalities])

    def encode(self, x):
        return self.encoder_net(x)

    def forward(self, x):
        z = self.encode(x)
        h = self.shared_decoder(z)
        return {
            "z": z,
            "cont": self.cont_head(h),
            "binary_logits": self.bin_head(h),
            "cat_logits": [head(h) for head in self.cat_heads],
        }


def mixed_reconstruction_loss(outputs, batch, weights=None, sparse_lambda=0.0):
    if weights is None:
        weights = {"cont": 1.0, "binary": 1.0, "cat": 1.0}

    cont = batch["cont"]
    cont_mask = batch["cont_mask"]
    cont_num = ((outputs["cont"] - cont) ** 2 * cont_mask).sum()
    cont_den = cont_mask.sum().clamp_min(1.0)
    cont_loss = cont_num / cont_den

    binary_loss = F.binary_cross_entropy_with_logits(outputs["binary_logits"], batch["binary"])

    cat_losses = []
    for j, logits in enumerate(outputs["cat_logits"]):
        cat_losses.append(F.cross_entropy(logits, batch["cat"][:, j]))
    cat_loss = torch.stack(cat_losses).mean()

    total = weights["cont"] * cont_loss + weights["binary"] * binary_loss + weights["cat"] * cat_loss
    if sparse_lambda > 0:
        total = total + sparse_lambda * torch.mean(torch.abs(outputs["z"]))

    return total, {"cont_loss": cont_loss, "binary_loss": binary_loss, "cat_loss": cat_loss}


def corrupt_input(x, corruption_rate=0.0, noise_std=0.0):
    out = x
    if corruption_rate > 0:
        keep = (torch.rand_like(out) > corruption_rate).float()
        out = out * keep
    if noise_std > 0:
        out = out + noise_std * torch.randn_like(out)
    return out


def train_mixed_autoencoder(
    model,
    train_loader,
    val_loader,
    epochs=TRAIN_EPOCHS,
    lr=1e-3,
    name="mixed_autoencoder",
    loss_weights=None,
    corruption_rate=0.0,
    noise_std=0.0,
    sparse_lambda=0.0,
):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        train_total, train_cont, train_bin, train_cat = [], [], [], []
        for batch in train_loader:
            batch = move_batch(batch)
            x_in = corrupt_input(batch["X"], corruption_rate=corruption_rate, noise_std=noise_std)
            optimizer.zero_grad()
            outputs = model(x_in)
            loss, parts = mixed_reconstruction_loss(outputs, batch, weights=loss_weights, sparse_lambda=sparse_lambda)
            loss.backward()
            optimizer.step()
            train_total.append(float(loss.detach().cpu()))
            train_cont.append(float(parts["cont_loss"].detach().cpu()))
            train_bin.append(float(parts["binary_loss"].detach().cpu()))
            train_cat.append(float(parts["cat_loss"].detach().cpu()))

        model.eval()
        val_total = []
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch)
                outputs = model(batch["X"])
                loss, _ = mixed_reconstruction_loss(outputs, batch, weights=loss_weights, sparse_lambda=sparse_lambda)
                val_total.append(float(loss.detach().cpu()))

        history.append({
            "epoch": epoch,
            "train_loss": np.mean(train_total),
            "train_cont": np.mean(train_cont),
            "train_binary": np.mean(train_bin),
            "train_cat": np.mean(train_cat),
            "val_loss": np.mean(val_total),
        })
        if epoch in [1, epochs] or epoch % max(1, epochs // 3) == 0:
            print(f"{name} epoch {epoch:02d} | train {np.mean(train_total):.4f} | val {np.mean(val_total):.4f}")

    return history


@torch.no_grad()
def encode_mixed_model(model, X, batch_size=512):
    model.eval()
    zs = []
    for start in range(0, len(X), batch_size):
        x = torch.as_tensor(X[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        z = model.encode(x)
        zs.append(z.detach().cpu().numpy())
    return np.vstack(zs)


@torch.no_grad()
def compute_mixed_reconstruction_errors(model, loader):
    model.eval()
    rows = []
    for batch in loader:
        row_indices = batch["row_index"].numpy()
        batch = move_batch(batch)
        outputs = model(batch["X"])

        cont_mask = batch["cont_mask"]
        cont_err = ((outputs["cont"] - batch["cont"]) ** 2 * cont_mask).sum(dim=1) / cont_mask.sum(dim=1).clamp_min(1.0)
        bin_err = F.binary_cross_entropy_with_logits(outputs["binary_logits"], batch["binary"], reduction="none").mean(dim=1)
        cat_err_parts = []
        for j, logits in enumerate(outputs["cat_logits"]):
            cat_err_parts.append(F.cross_entropy(logits, batch["cat"][:, j], reduction="none"))
        cat_err = torch.stack(cat_err_parts, dim=1).mean(dim=1)
        total_err = cont_err + bin_err + cat_err

        for i, row_idx in enumerate(row_indices):
            rows.append({
                "row_index": int(row_idx),
                "cont_error": float(cont_err[i].detach().cpu()),
                "binary_error": float(bin_err[i].detach().cpu()),
                "cat_error": float(cat_err[i].detach().cpu()),
                "total_error": float(total_err[i].detach().cpu()),
            })
    return pd.DataFrame(rows).sort_values("row_index").reset_index(drop=True)

In [ ]:
# ============================================================
# 6.3 Train the mixed-type autoencoder
# ============================================================
cat_cardinalities = [len(CAT_LEVELS[col]) for col in CATEGORICAL_FEATURES]

mixed_ae = MixedTypeAutoencoder(
    input_dim=X_rep.shape[1],
    n_cont=len(CONTINUOUS_FEATURES),
    n_binary=len(BINARY_FEATURES),
    cat_cardinalities=cat_cardinalities,
    latent_dim=LATENT_DIM,
)

mixed_history = train_mixed_autoencoder(
    mixed_ae,
    train_loader,
    val_loader,
    epochs=TRAIN_EPOCHS,
    lr=1e-3,
    name="Mixed-type AE",
)
plot_history(mixed_history, "Mixed-type autoencoder loss")

Z_mixed = encode_mixed_model(mixed_ae, X_rep)
mixed_metrics, mixed_prob, mixed_downstream_model = evaluate_binary_model_from_features(
    Z_mixed, y_churn, train_idx, val_idx, test_idx, name="mixed_AE_embedding_logistic"
)
representation_results.append(mixed_metrics)

mixed_errors_all = compute_mixed_reconstruction_errors(mixed_ae, all_loader)
print("Mixed AE downstream churn metrics:")
display(pd.DataFrame([mixed_metrics]))
print("Mixed AE reconstruction error components:")
display(mixed_errors_all[["cont_error", "binary_error", "cat_error", "total_error"]].describe().T)
plot_embedding_2d(Z_mixed, labels=model_df["customer_archetype"], title="Mixed-type autoencoder embeddings by archetype")

## 7. Embeddings for downstream prediction and segmentation

An embedding is useful only if it improves decisions or makes decision workflows more stable. Here we evaluate frozen embeddings on a churn anchor task, then use the same embeddings for customer segmentation. The cluster descriptions are computed after clustering so the segment labels remain business-readable.

In [ ]:
# ============================================================
# 7.1 Compare representation choices on the churn anchor task
# ============================================================
representation_results_df = summarize_results(representation_results)
display(representation_results_df)

plt.figure(figsize=(8, 4))
plt.bar(representation_results_df["model"], representation_results_df["roc_auc"])
plt.xticks(rotation=30, ha="right")
plt.ylim(0, 1.05)
plt.ylabel("Test ROC-AUC")
plt.title("Frozen representations evaluated on churn prediction")
plt.tight_layout()
plt.show()

plot_confusion_matrix(y_churn[test_idx], mixed_prob, threshold=mixed_metrics["threshold"], title="Mixed AE embedding churn model")

In [ ]:
# ============================================================
# 7.2 Customer segmentation from embeddings
# ============================================================
K = 5
kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=20)
cluster_train = kmeans.fit_predict(Z_mixed[train_idx])
clusters_all = kmeans.predict(Z_mixed)
model_df["embedding_cluster"] = clusters_all

cluster_profile = (
    model_df.groupby("embedding_cluster")
    .agg(
        n=("customer_id", "size"),
        churn_rate=("churn_next_60d", "mean"),
        cross_sell_rate=("cross_sell_next_60d", "mean"),
        median_spend=("monthly_spend", "median"),
        median_recency=("recency_days", "median"),
        complaint_rate=("recent_complaint", "mean"),
        loyalty_rate=("loyalty_member", "mean"),
        coupon_rate=("coupon_redeemer", "mean"),
    )
    .reset_index()
)

display(cluster_profile)
print("Archetype by learned cluster:")
display(pd.crosstab(model_df["embedding_cluster"], model_df["customer_archetype"], normalize="index").round(3))

try:
    sil = silhouette_score(Z_mixed[test_idx], clusters_all[test_idx])
    print("Test silhouette score using mixed AE embeddings:", round(float(sil), 3))
except Exception as exc:
    print("Silhouette score skipped:", exc)

## 8. Practical variants: denoising and sparsity

Autoencoder variants keep the same encode-decode blueprint but change the constraint. A denoising autoencoder corrupts the input and reconstructs the clean target, which encourages robustness. A sparse autoencoder adds a penalty that pushes most latent activations toward zero, which can make profiles more factor-like. Contractive autoencoders add a stronger stability penalty through encoder derivatives, but that is usually too heavy for an introductory Colab. We use perturbation stability as a practical diagnostic instead.

In [ ]:
# ============================================================
# 8.1 Train denoising and sparse variants
# - Reduce or skip this cell if runtime is tight.
# ============================================================
RUN_VARIANT_COMPARISON = True

variant_rows = []

def latent_activation_summary(Z, name):
    return {
        "variant": name,
        "mean_abs_z": float(np.mean(np.abs(Z))),
        "median_abs_z": float(np.median(np.abs(Z))),
        "share_near_zero_abs_lt_0_05": float((np.abs(Z) < 0.05).mean()),
        "mean_latent_std": float(np.std(Z, axis=0).mean()),
    }

if RUN_VARIANT_COMPARISON:
    denoise_ae = MixedTypeAutoencoder(
        input_dim=X_rep.shape[1],
        n_cont=len(CONTINUOUS_FEATURES),
        n_binary=len(BINARY_FEATURES),
        cat_cardinalities=cat_cardinalities,
        latent_dim=LATENT_DIM,
    )
    denoise_history = train_mixed_autoencoder(
        denoise_ae,
        train_loader,
        val_loader,
        epochs=VARIANT_EPOCHS,
        lr=1e-3,
        name="Denoising AE",
        corruption_rate=0.12,
        noise_std=0.03,
    )
    Z_denoise = encode_mixed_model(denoise_ae, X_rep)
    denoise_metrics, denoise_prob, _ = evaluate_binary_model_from_features(
        Z_denoise, y_churn, train_idx, val_idx, test_idx, name="denoising_AE_embedding_logistic"
    )
    representation_results.append(denoise_metrics)

    sparse_ae = MixedTypeAutoencoder(
        input_dim=X_rep.shape[1],
        n_cont=len(CONTINUOUS_FEATURES),
        n_binary=len(BINARY_FEATURES),
        cat_cardinalities=cat_cardinalities,
        latent_dim=LATENT_DIM,
    )
    sparse_history = train_mixed_autoencoder(
        sparse_ae,
        train_loader,
        val_loader,
        epochs=VARIANT_EPOCHS,
        lr=1e-3,
        name="Sparse AE",
        sparse_lambda=0.003,
    )
    Z_sparse = encode_mixed_model(sparse_ae, X_rep)
    sparse_metrics, sparse_prob, _ = evaluate_binary_model_from_features(
        Z_sparse, y_churn, train_idx, val_idx, test_idx, name="sparse_AE_embedding_logistic"
    )
    representation_results.append(sparse_metrics)

    variant_rows = [
        latent_activation_summary(Z_mixed, "mixed_AE"),
        latent_activation_summary(Z_denoise, "denoising_AE"),
        latent_activation_summary(Z_sparse, "sparse_AE"),
    ]
    display(pd.DataFrame(variant_rows))
else:
    print("Variant comparison skipped.")

In [ ]:
# ============================================================
# 8.2 Perturbation stability diagnostic
# ============================================================
@torch.no_grad()
def perturbation_stability(model, X, sample_idx, corruption_rate=0.10, noise_std=0.03, repeats=5):
    model.eval()
    x_clean = torch.as_tensor(X[sample_idx], dtype=torch.float32, device=DEVICE)
    z_clean = model.encode(x_clean).detach().cpu().numpy()
    distances = []
    for _ in range(repeats):
        x_noisy = corrupt_input(x_clean, corruption_rate=corruption_rate, noise_std=noise_std)
        z_noisy = model.encode(x_noisy).detach().cpu().numpy()
        distances.append(np.sqrt(((z_noisy - z_clean) ** 2).sum(axis=1)))
    return np.vstack(distances).mean(axis=0)

rng = np.random.default_rng(SEED)
stability_idx = rng.choice(test_idx, size=min(400, len(test_idx)), replace=False)

stability_rows = []
stability_rows.append({
    "model": "mixed_AE",
    "mean_embedding_shift": float(perturbation_stability(mixed_ae, X_rep, stability_idx).mean()),
    "median_embedding_shift": float(np.median(perturbation_stability(mixed_ae, X_rep, stability_idx))),
})
if RUN_VARIANT_COMPARISON:
    stability_rows.append({
        "model": "denoising_AE",
        "mean_embedding_shift": float(perturbation_stability(denoise_ae, X_rep, stability_idx).mean()),
        "median_embedding_shift": float(np.median(perturbation_stability(denoise_ae, X_rep, stability_idx))),
    })
    stability_rows.append({
        "model": "sparse_AE",
        "mean_embedding_shift": float(perturbation_stability(sparse_ae, X_rep, stability_idx).mean()),
        "median_embedding_shift": float(np.median(perturbation_stability(sparse_ae, X_rep, stability_idx))),
    })

stability_df = pd.DataFrame(stability_rows)
display(stability_df)

plt.figure(figsize=(7, 4))
plt.bar(stability_df["model"], stability_df["mean_embedding_shift"])
plt.ylabel("Mean L2 shift after input perturbation")
plt.title("Practical stability check for representation variants")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

## 9. Variational autoencoder (VAE)

A variational autoencoder maps each input to a distribution over latent codes rather than a single point. The encoder produces `mu` and `logvar`, a latent code is sampled using the reparameterization trick, and the decoder reconstructs the input. The loss balances reconstruction with a regularization term that keeps the latent space smooth. In this chapter, the VAE is used primarily as a representation-learning model and as a bridge to generative modeling.

For a compact classroom visualization, the VAE below uses a two-dimensional latent space and an MSE reconstruction loss over the preprocessed vector.

In [ ]:
# ============================================================
# 9.1 Simple VAE model and training helper
# ============================================================
class SimpleVAE(nn.Module):
    def __init__(self, input_dim, latent_dim=2, hidden_dim=64):
        super().__init__()
        self.encoder_body = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.ReLU(),
        )
        self.mu_head = nn.Linear(hidden_dim // 2, latent_dim)
        self.logvar_head = nn.Linear(hidden_dim // 2, latent_dim)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim // 2),
            nn.ReLU(),
            nn.Linear(hidden_dim // 2, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, input_dim),
        )

    def encode_stats(self, x):
        h = self.encoder_body(x)
        return self.mu_head(h), self.logvar_head(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def encode(self, x):
        mu, _ = self.encode_stats(x)
        return mu

    def forward(self, x):
        mu, logvar = self.encode_stats(x)
        z = self.reparameterize(mu, logvar)
        x_hat = self.decoder(z)
        return x_hat, mu, logvar, z


def vae_loss(x_hat, x, mu, logvar, beta=1.0):
    recon = F.mse_loss(x_hat, x, reduction="mean")
    kl = -0.5 * torch.mean(torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1))
    return recon + beta * kl, recon, kl


def train_vae(model, train_loader, val_loader, epochs=VAE_EPOCHS, lr=1e-3, beta_max=0.6, anneal_epochs=6):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    history = []

    for epoch in range(1, epochs + 1):
        beta = beta_max * min(1.0, epoch / max(1, anneal_epochs))
        model.train()
        train_total, train_recon, train_kl = [], [], []
        for batch in train_loader:
            batch = move_batch(batch)
            x = batch["X"]
            optimizer.zero_grad()
            x_hat, mu, logvar, _ = model(x)
            loss, recon, kl = vae_loss(x_hat, x, mu, logvar, beta=beta)
            loss.backward()
            optimizer.step()
            train_total.append(float(loss.detach().cpu()))
            train_recon.append(float(recon.detach().cpu()))
            train_kl.append(float(kl.detach().cpu()))

        model.eval()
        val_total = []
        with torch.no_grad():
            for batch in val_loader:
                batch = move_batch(batch)
                x = batch["X"]
                x_hat, mu, logvar, _ = model(x)
                loss, _, _ = vae_loss(x_hat, x, mu, logvar, beta=beta)
                val_total.append(float(loss.detach().cpu()))

        history.append({
            "epoch": epoch,
            "beta": beta,
            "train_loss": np.mean(train_total),
            "train_recon": np.mean(train_recon),
            "train_kl": np.mean(train_kl),
            "val_loss": np.mean(val_total),
        })
        if epoch in [1, epochs] or epoch % max(1, epochs // 3) == 0:
            print(f"VAE epoch {epoch:02d} | beta {beta:.2f} | train {np.mean(train_total):.4f} | val {np.mean(val_total):.4f}")
    return history


@torch.no_grad()
def encode_vae(model, X, batch_size=512):
    model.eval()
    zs = []
    for start in range(0, len(X), batch_size):
        x = torch.as_tensor(X[start:start + batch_size], dtype=torch.float32, device=DEVICE)
        z = model.encode(x)
        zs.append(z.detach().cpu().numpy())
    return np.vstack(zs)

In [ ]:
# ============================================================
# 9.2 Train the VAE and inspect its two-dimensional latent space
# ============================================================
vae = SimpleVAE(input_dim=X_rep.shape[1], latent_dim=2)
vae_history = train_vae(vae, train_loader, val_loader, epochs=VAE_EPOCHS, lr=1e-3, beta_max=0.6, anneal_epochs=6)
plot_history(vae_history, "VAE training with beta annealing")

Z_vae = encode_vae(vae, X_rep)
vae_metrics, vae_prob, vae_downstream_model = evaluate_binary_model_from_features(
    Z_vae, y_churn, train_idx, val_idx, test_idx, name="VAE_mu_embedding_logistic"
)
representation_results.append(vae_metrics)

print("VAE downstream churn metrics:")
display(pd.DataFrame([vae_metrics]))
plot_embedding_2d(Z_vae, labels=model_df["customer_archetype"], title="VAE latent means by archetype")

In [ ]:
# ============================================================
# 9.3 Decode a few random latent points as a generative intuition check
# ============================================================
@torch.no_grad()
def decode_vae_samples(model, n=5, seed=SEED):
    rng = np.random.default_rng(seed)
    z = torch.as_tensor(rng.normal(size=(n, 2)), dtype=torch.float32, device=DEVICE)
    x_hat = model.decoder(z).detach().cpu().numpy()
    cont_slice = arrays["slices"]["continuous_scaled"]
    bin_slice = arrays["slices"]["binary"]
    cont_scaled_hat = x_hat[:, cont_slice]
    cont_unscaled = preprocessor["scaler"].inverse_transform(cont_scaled_hat)
    out = pd.DataFrame(cont_unscaled, columns=CONTINUOUS_FEATURES)
    # Convert log-transformed columns back to their original scale for readability.
    for raw_col in LOG_TRANSFORM:
        feat_col = f"log_{raw_col}"
        if feat_col in out.columns:
            out[raw_col] = np.expm1(out[feat_col]).clip(lower=0)
    bin_logits = x_hat[:, bin_slice]
    bin_prob = 1 / (1 + np.exp(-bin_logits))
    for j, col in enumerate(BINARY_FEATURES):
        out[f"prob_{col}"] = bin_prob[:, j]
    keep_cols = [col for col in ["monthly_spend", "purchase_count", "session_count", "recency_days", "prob_loyalty_member", "prob_recent_complaint"] if col in out.columns]
    return out[keep_cols].round(3)

print("Decoded random latent points are not production-quality synthetic data.")
print("They are a classroom check that the VAE latent space can be sampled and decoded.")
display(decode_vae_samples(vae, n=6))

## 10. Embedding-space geometry and nearest-neighbor search

Once an encoder produces embeddings, business users often ask for similar customers, similar products, or similar sessions. Similarity is not universal. Euclidean distance is sensitive to magnitude, cosine similarity focuses more on direction, and Mahalanobis distance accounts for covariance structure in the embedding space. For large production systems, approximate nearest-neighbor indexes are commonly used. Here exact search is enough for classroom inspection.

In [ ]:
# ============================================================
# 10.1 Compare Euclidean, cosine, and Mahalanobis neighbors
# ============================================================
Z_lookup = Z_mixed.copy()
lookup_pool_idx = np.concatenate([train_idx, val_idx])
query_candidates = test_idx[y_churn[test_idx] == 1]
query_idx = int(query_candidates[0] if len(query_candidates) else test_idx[0])


def neighbor_table(metric_name, neighbor_indices, distances):
    rows = []
    for rank, (idx, dist) in enumerate(zip(neighbor_indices, distances), start=1):
        rows.append({
            "rank": rank,
            "distance": float(dist),
            "customer_id": int(model_df.loc[idx, "customer_id"]),
            "archetype": model_df.loc[idx, "customer_archetype"],
            "cluster": int(model_df.loc[idx, "embedding_cluster"]),
            "monthly_spend": float(model_df.loc[idx, "monthly_spend"]),
            "recency_days": int(model_df.loc[idx, "recency_days"]),
            "recent_complaint": int(model_df.loc[idx, "recent_complaint"]),
            "churn_next_60d": int(model_df.loc[idx, "churn_next_60d"]),
            "metric": metric_name,
        })
    return pd.DataFrame(rows)

query_z = Z_lookup[query_idx:query_idx + 1]

nn_euclid = NearestNeighbors(n_neighbors=6, metric="euclidean")
nn_euclid.fit(Z_lookup[lookup_pool_idx])
dist_e, pos_e = nn_euclid.kneighbors(query_z)
idx_e = lookup_pool_idx[pos_e[0]]

def cosine_neighbors(Z, query_z, pool_idx, k=6):
    nn = NearestNeighbors(n_neighbors=k, metric="cosine")
    nn.fit(Z[pool_idx])
    dist, pos = nn.kneighbors(query_z)
    return pool_idx[pos[0]], dist[0]

idx_c, dist_c = cosine_neighbors(Z_lookup, query_z, lookup_pool_idx, k=6)

cov = np.cov(Z_lookup[train_idx].T)
VI = np.linalg.pinv(cov + 1e-4 * np.eye(cov.shape[0]))
diff = Z_lookup[lookup_pool_idx] - query_z
maha_dist = np.sqrt(np.einsum("ij,jk,ik->i", diff, VI, diff))
maha_order = np.argsort(maha_dist)[:6]
idx_m = lookup_pool_idx[maha_order]
dist_m = maha_dist[maha_order]

query_profile = model_df.loc[query_idx, ["customer_id", "customer_archetype", "monthly_spend", "recency_days", "recent_complaint", "churn_next_60d"]]
print_section("Query customer")
display(query_profile.to_frame("value"))

neighbors_df = pd.concat([
    neighbor_table("euclidean", idx_e, dist_e[0]),
    neighbor_table("cosine", idx_c, dist_c),
    neighbor_table("mahalanobis", idx_m, dist_m),
], ignore_index=True)
display(neighbors_df)

In [ ]:
# ============================================================
# 10.2 Visualize the nearest-neighbor neighborhood
# ============================================================
neighbor_ids = sorted(set(neighbors_df["customer_id"].astype(int).tolist()))
neighbor_row_idx = model_df.index[model_df["customer_id"].isin(neighbor_ids)].to_numpy()
plot_idx = np.concatenate([[query_idx], neighbor_row_idx])
Z2_local = PCA(n_components=2, random_state=SEED).fit_transform(Z_lookup[np.concatenate([lookup_pool_idx, [query_idx]])])
# Build a small local view by projecting all pool records plus the query, then showing only neighbors.
pool_plus_query = np.concatenate([lookup_pool_idx, [query_idx]])
local_df = pd.DataFrame(Z2_local, columns=["axis1", "axis2"])
local_df["row_index"] = pool_plus_query
local_df["role"] = np.where(local_df["row_index"] == query_idx, "query", "pool")
local_neighbors = local_df[local_df["row_index"].isin(plot_idx)]

plt.figure(figsize=(6, 5))
plt.scatter(local_df["axis1"], local_df["axis2"], s=8, alpha=0.15, label="lookup pool")
plt.scatter(local_neighbors["axis1"], local_neighbors["axis2"], s=60, alpha=0.90, label="query and neighbors")
plt.xlabel("PCA axis 1 of mixed AE embedding")
plt.ylabel("PCA axis 2 of mixed AE embedding")
plt.title("Query customer and retrieved neighbors")
plt.legend()
plt.tight_layout()
plt.show()

## 11. Reconstruction error as an anomaly score

Autoencoders provide a second useful output: reconstruction error. If the model is trained mostly on typical patterns, unusual records often reconstruct poorly. The error is not a true label by itself. It is a prioritization score that must be calibrated against review capacity and business cost.

In [ ]:
# ============================================================
# 11.1 Use mixed-type reconstruction error for anomaly triage
# ============================================================
error_df = mixed_errors_all.copy()
error_df = error_df.merge(
    model_df[["customer_id", "as_of_month", "customer_archetype", "anomaly_label", "monthly_spend", "support_minutes", "return_rate"]],
    left_on="row_index",
    right_index=True,
    how="left",
)

val_threshold = float(error_df.loc[error_df["row_index"].isin(val_idx), "total_error"].quantile(0.95))
test_errors = error_df[error_df["row_index"].isin(test_idx)].copy()
test_errors["alert"] = (test_errors["total_error"] >= val_threshold).astype(int)

alert_precision = test_errors.loc[test_errors["alert"] == 1, "anomaly_label"].mean() if test_errors["alert"].sum() > 0 else np.nan
alert_recall = test_errors.loc[test_errors["anomaly_label"] == 1, "alert"].mean() if test_errors["anomaly_label"].sum() > 0 else np.nan
anomaly_auc = safe_auc(test_errors["anomaly_label"].to_numpy(), test_errors["total_error"].to_numpy())

anomaly_summary = pd.DataFrame([{
    "threshold_source": "95th percentile of validation reconstruction error",
    "threshold": val_threshold,
    "test_alert_rate": float(test_errors["alert"].mean()),
    "precision_among_alerts": float(alert_precision),
    "recall_of_injected_anomalies": float(alert_recall),
    "anomaly_roc_auc": float(anomaly_auc),
}])
display(anomaly_summary)

print("Highest reconstruction-error test records:")
display(test_errors.sort_values("total_error", ascending=False).head(10))

In [ ]:
# ============================================================
# 11.2 Histogram with alert threshold
# ============================================================
plt.figure(figsize=(8, 4))
plt.hist(test_errors["total_error"], bins=40)
plt.axvline(val_threshold, linestyle="--", linewidth=2, label="validation 95th percentile threshold")
plt.xlabel("Mixed reconstruction error")
plt.ylabel("Number of test records")
plt.title("Reconstruction error as an anomaly score")
plt.legend()
plt.tight_layout()
plt.show()

plt.figure(figsize=(7, 4))
plot_data = test_errors.copy()
plot_data["group"] = np.where(plot_data["anomaly_label"] == 1, "injected anomaly", "typical")
for group in ["typical", "injected anomaly"]:
    vals = plot_data.loc[plot_data["group"] == group, "total_error"]
    plt.hist(vals, bins=30, alpha=0.55, label=group)
plt.xlabel("Mixed reconstruction error")
plt.ylabel("Number of test records")
plt.title("Do injected anomalies reconstruct poorly?")
plt.legend()
plt.tight_layout()
plt.show()

## 12. Evaluation, debugging, and governance

Representation learning can make a team feel productive because reconstruction loss almost always decreases. Business acceptance requires more evidence: downstream lift against baselines, stability under realistic perturbations, absence of collapse, subgroup checks, time integrity, and documented intended use.

In [ ]:
# ============================================================
# 12.1 Representation diagnostics
# ============================================================
def embedding_diagnostics(Z, name, sample_size=800):
    rng = np.random.default_rng(SEED)
    idx = np.arange(Z.shape[0])
    if len(idx) > sample_size:
        idx = rng.choice(idx, size=sample_size, replace=False)
    Zs = Z[idx]
    pair_idx_1 = rng.choice(np.arange(len(Zs)), size=min(2000, len(Zs) * 2), replace=True)
    pair_idx_2 = rng.choice(np.arange(len(Zs)), size=min(2000, len(Zs) * 2), replace=True)
    l2 = np.sqrt(((Zs[pair_idx_1] - Zs[pair_idx_2]) ** 2).sum(axis=1))
    return {
        "embedding": name,
        "n_dimensions": Z.shape[1],
        "mean_abs_value": float(np.mean(np.abs(Z))),
        "mean_dimension_std": float(np.std(Z, axis=0).mean()),
        "min_dimension_std": float(np.std(Z, axis=0).min()),
        "median_pairwise_l2": float(np.median(l2)),
        "collapse_warning": bool(np.std(Z, axis=0).mean() < 0.02),
    }

embedding_diag_rows = [
    embedding_diagnostics(Z_pca, "PCA"),
    embedding_diagnostics(Z_basic, "basic_AE"),
    embedding_diagnostics(Z_mixed, "mixed_AE"),
    embedding_diagnostics(Z_vae, "VAE_mu"),
]
if RUN_VARIANT_COMPARISON:
    embedding_diag_rows.extend([
        embedding_diagnostics(Z_denoise, "denoising_AE"),
        embedding_diagnostics(Z_sparse, "sparse_AE"),
    ])
embedding_diagnostics_df = pd.DataFrame(embedding_diag_rows)
display(embedding_diagnostics_df)

representation_results_df = summarize_results(representation_results)
display(representation_results_df)

In [ ]:
# ============================================================
# 12.2 Subgroup behavior check on the anchor churn task
# ============================================================
subgroup_rows = []
churn_pred_mixed = (mixed_prob >= mixed_metrics["threshold"]).astype(int)
test_frame = model_df.iloc[test_idx].copy()
test_frame["mixed_prob"] = mixed_prob
test_frame["mixed_pred"] = churn_pred_mixed

for group_name, g in test_frame.groupby("customer_archetype"):
    subgroup_rows.append({
        "customer_archetype": group_name,
        "n": len(g),
        "churn_rate": float(g["churn_next_60d"].mean()),
        "pred_positive_rate": float(g["mixed_pred"].mean()),
        "roc_auc": safe_auc(g["churn_next_60d"].to_numpy(), g["mixed_prob"].to_numpy()),
        "f1": f1_score(g["churn_next_60d"], g["mixed_pred"], zero_division=0),
    })

subgroup_df = pd.DataFrame(subgroup_rows).sort_values("roc_auc", ascending=False)
display(subgroup_df)

In [ ]:
# ============================================================
# 12.3 Failure-mode and governance checklists
# ============================================================
failure_modes = pd.DataFrame([
    {
        "failure_mode": "Volume dominates the embedding",
        "what_you_observe": "Clusters mostly reflect spend or activity level",
        "likely_cause": "Scaling or loss weighting allows large-range features to dominate",
        "practical_fix": "Rescale continuous features and audit feature-group reconstruction errors",
    },
    {
        "failure_mode": "Missing equals zero artifacts",
        "what_you_observe": "Segments align with missingness or logging regimes",
        "likely_cause": "Missingness was not modeled separately",
        "practical_fix": "Use missingness masks and indicators, and design realistic corruption",
    },
    {
        "failure_mode": "Copying or memorization",
        "what_you_observe": "Very low loss but weak downstream lift",
        "likely_cause": "Bottleneck is too weak or model capacity is too high",
        "practical_fix": "Strengthen the bottleneck, add noise, add sparsity, or validate against PCA",
    },
    {
        "failure_mode": "Collapse",
        "what_you_observe": "Many inputs map to nearly identical embeddings",
        "likely_cause": "Optimization shortcuts or overly strong constraints",
        "practical_fix": "Check latent variance, relax constraints, and revisit learning rate",
    },
    {
        "failure_mode": "Offline success, online failure",
        "what_you_observe": "Experiment metrics look good but production KPIs do not improve",
        "likely_cause": "Leakage, unrealistic splits, or mismatch with decision context",
        "practical_fix": "Use as-of timestamps, time-respecting splits, and decision-aligned evaluation",
    },
])

governance_checklist = pd.DataFrame([
    {
        "governance_area": "Intended use",
        "question_to_ask": "What decisions will use this embedding, and which will not?",
        "evidence": "Written intended-use statement and explicit exclusions",
    },
    {
        "governance_area": "Time integrity",
        "question_to_ask": "Does every embedding have a clear as-of timestamp?",
        "evidence": "Entity ID, encoder version, preprocessing version, and as-of time stored together",
    },
    {
        "governance_area": "Versioning",
        "question_to_ask": "Can last quarter's experiment be reproduced?",
        "evidence": "Encoder version, data window, feature list, and training parameters recorded",
    },
    {
        "governance_area": "Data health",
        "question_to_ask": "Are upstream inputs stable and consistent?",
        "evidence": "Monitoring of missingness, ranges, category levels, and pipeline changes",
    },
    {
        "governance_area": "Downstream value",
        "question_to_ask": "Does the embedding still improve decisions relative to baselines?",
        "evidence": "Periodic revalidation on anchor tasks against PCA and engineered features",
    },
    {
        "governance_area": "Privacy and sensitivity",
        "question_to_ask": "Could the embedding proxy sensitive information?",
        "evidence": "Access limits, subgroup reporting, and documented risk assessment",
    },
])

print("Common failure modes:")
display(failure_modes)
print("Governance checklist:")
display(governance_checklist)

In [ ]:
# ============================================================
# 12.4 Create a lightweight representation card
# ============================================================
representation_card = {
    "entity": "customer",
    "as_of_field": "as_of_month",
    "intended_use": [
        "customer segmentation",
        "similarity search for lookalike analysis",
        "frozen embedding features for churn and cross-sell anchor tasks",
        "reconstruction-error triage for unusual records",
    ],
    "explicit_exclusions": [
        "high-impact credit, employment, or eligibility decisions",
        "uses that require protected-attribute inference",
        "reuse after task-specific fine-tuning without a new version",
    ],
    "preprocessing_version": "ch17_synthetic_preprocessor_v1",
    "encoder_version": "mixed_AE_v1_demo",
    "latent_dimension": LATENT_DIM,
    "training_window": {
        "train_as_of_month_max": int(model_df.iloc[train_idx]["as_of_month"].max()),
        "validation_as_of_month_range": [
            int(model_df.iloc[val_idx]["as_of_month"].min()),
            int(model_df.iloc[val_idx]["as_of_month"].max()),
        ],
        "test_as_of_month_min": int(model_df.iloc[test_idx]["as_of_month"].min()),
    },
    "input_groups": {
        "continuous": CONTINUOUS_FEATURES,
        "binary": BINARY_FEATURES,
        "categorical": CATEGORICAL_FEATURES,
    },
    "refresh_guidance": "Regenerate embeddings from the fixed encoder regularly. Retrain the encoder only when data health, drift, or anchor-task performance indicates need.",
}

print(json.dumps(representation_card, indent=2))

In [ ]:
# ============================================================
# 12.5 Save artifacts
# ============================================================
embedding_cols = [f"z_{j:02d}" for j in range(Z_mixed.shape[1])]
embedding_df = pd.DataFrame(Z_mixed, columns=embedding_cols)
embedding_df.insert(0, "customer_id", model_df["customer_id"].values)
embedding_df.insert(1, "as_of_month", model_df["as_of_month"].values)
embedding_df.insert(2, "customer_archetype", model_df["customer_archetype"].values)
embedding_df.insert(3, "embedding_cluster", model_df["embedding_cluster"].values)

customers_df.to_csv(OUT_DIR / "ch17_synthetic_customers.csv", index=False)
embedding_df.to_csv(OUT_DIR / "ch17_mixed_ae_embeddings.csv", index=False)
representation_results_df.to_csv(OUT_DIR / "ch17_representation_results.csv", index=False)
error_df.to_csv(OUT_DIR / "ch17_reconstruction_error_scores.csv", index=False)
governance_checklist.to_csv(OUT_DIR / "ch17_governance_checklist.csv", index=False)

with open(OUT_DIR / "ch17_representation_card.json", "w") as f:
    json.dump(representation_card, f, indent=2)

print("Saved artifacts:")
for file_name in [
    "ch17_synthetic_customers.csv",
    "ch17_mixed_ae_embeddings.csv",
    "ch17_representation_results.csv",
    "ch17_reconstruction_error_scores.csv",
    "ch17_governance_checklist.csv",
    "ch17_representation_card.json",
]:
    print(" -", OUT_DIR / file_name)

## Exercises

1. Change `LATENT_DIM` from 8 to 2, 4, or 16, then rerun the notebook. Does a larger latent space improve reconstruction, downstream churn prediction, or segmentation stability?

2. Increase the denoising corruption rate from `0.12` to `0.25`. Does perturbation stability improve, and does the churn anchor task become better or worse?

3. In the mixed-type loss, change the weights so categorical reconstruction receives twice as much weight. Which customer clusters change most?

4. Replace the time-respecting split with a random split. Compare the results and explain why a random split can be misleading for reusable business embeddings.

5. Use `y_cross_sell` instead of `y_churn` as the anchor task. Which representation is most useful for cross-sell, and is it the same representation that works best for churn?

6. Choose a high-error test customer from the anomaly section. Compare its nearest neighbors under Euclidean and cosine distance. Would you send this record to manual review?

7. Add one additional governance field to the representation card. For example, record who owns the encoder, which systems can access the embeddings, or when retraining should be escalated.

## Wrap-up

This notebook showed the full representation-learning workflow for Chapter 17: mixed-type customer data, time-respecting preprocessing, PCA and engineered baselines, a basic autoencoder, a mixed-type decoder, denoising and sparsity constraints, a small VAE, segmentation, similarity search, anomaly scoring, diagnostics, and governance. The main lesson is stable across applications: an embedding is valuable only when it improves a business decision under a disciplined evaluation and monitoring process.